# Stage 9 — Difference-in-Differences (DiD) Analysis

This notebook estimates the effect of streetlight outages on crime using
the balanced causal panel created in Stage 8.

Outcomes:
- Crime within 0–100m
- Crime within 100–250m displacement ring

Treatment:
- 1 = outage location
- 0 = matched control location

Periods:
- PRE
- DURING
- POST

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.formula.api as smf

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
from pathlib import Path
import pandas as pd

PROCESSED_DIR = Path("../data/processed")
CAUSAL_FILE = PROCESSED_DIR / "causal_panel.parquet"

panel = pd.read_parquet(CAUSAL_FILE)

print("Causal panel loaded successfully.")
print("Shape:", panel.shape)
print("\nColumns:")
print(panel.columns.tolist())

Causal panel loaded successfully.
Shape: (276744, 16)

Columns:
['pair_id', 'location_key', 'created_date', 'closed_date', 'period', 'period_order', 'treatment', 'post', 'treatment_x_post', 'crime_100m', 'crime_250m', 'baseline_crime_intensity', 'borough', 'outage_duration_hours', 'h3_res9', 'h3_res10']


In [4]:
print("Treatment counts:")
print(panel["treatment"].value_counts().sort_index())

print("\nPeriod counts:")
print(panel["period"].value_counts())

print("\nTreatment × Period:")
print(
    pd.crosstab(
        panel["treatment"],
        panel["period"]
    )
)

Treatment counts:
treatment
0    138372
1    138372
Name: count, dtype: int64

Period counts:
period
pre       92248
during    92248
post      92248
Name: count, dtype: int64

Treatment × Period:
period     during   post    pre
treatment                      
0           46124  46124  46124
1           46124  46124  46124


In [5]:
# Verify DiD indicator variables

print("Treatment values:")
print(sorted(panel["treatment"].dropna().unique()))

print("\nPost values:")
print(sorted(panel["post"].dropna().unique()))

print("\nTreatment × Post values:")
print(sorted(panel["treatment_x_post"].dropna().unique()))

print("\nTreatment × Post crosstab:")
print(
    pd.crosstab(
        [panel["treatment"], panel["period"]],
        panel["treatment_x_post"]
    )
)

Treatment values:
[np.int64(0), np.int64(1)]

Post values:
[np.int64(0), np.int64(1)]

Treatment × Post values:
[np.int64(0), np.int64(1)]

Treatment × Post crosstab:
treatment_x_post      0      1
treatment period              
0         during  46124      0
          post    46124      0
          pre     46124      0
1         during      0  46124
          post        0  46124
          pre     46124      0


In [6]:
# Check crime outcome variables

print("Crime 0–100m summary:")
print(panel["crime_100m"].describe())

print("\nCrime 100–250m summary:")
print(panel["crime_250m"].describe())

print("\nMissing values:")
print(
    panel[["crime_100m", "crime_250m"]]
    .isna()
    .sum()
)

print("\nNumber of non-zero observations:")
print(
    "crime_100m:",
    (panel["crime_100m"] > 0).sum()
)

print(
    "crime_250m:",
    (panel["crime_250m"] > 0).sum()
)

Crime 0–100m summary:
count    276744.000000
mean          0.005915
std           0.142395
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          48.000000
Name: crime_100m, dtype: float64

Crime 100–250m summary:
count    276744.000000
mean          0.027310
std           0.432036
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          67.000000
Name: crime_250m, dtype: float64

Missing values:
crime_100m    0
crime_250m    0
dtype: int64

Number of non-zero observations:
crime_100m: 1218
crime_250m: 4488


In [7]:
# Descriptive crime means by treatment group and period

mean_table = (
    panel
    .groupby(["treatment", "period"])[["crime_100m", "crime_250m"]]
    .mean()
    .reset_index()
)

print("Mean crime counts by treatment group and period:")
print(mean_table.to_string(index=False))

Mean crime counts by treatment group and period:
 treatment period  crime_100m  crime_250m
         0 during    0.008759    0.036510
         0   post    0.005117    0.022440
         0    pre    0.006071    0.023372
         1 during    0.006157    0.035621
         1   post    0.004878    0.021919
         1    pre    0.004510    0.024001


In [8]:
# Calculate simple changes from PRE to POST

pivot_means = (
    panel
    .groupby(["treatment", "period"])[["crime_100m", "crime_250m"]]
    .mean()
    .unstack("period")
)

print("Mean crime change: PRE → POST")
print(pivot_means)

Mean crime change: PRE → POST
          crime_100m                     crime_250m                    
period        during      post       pre     during      post       pre
treatment                                                              
0           0.008759  0.005117  0.006071   0.036510  0.022440  0.023372
1           0.006157  0.004878  0.004510   0.035621  0.021919  0.024001


In [9]:
# Difference-in-Differences model for 0–100m crime

did_100m = smf.ols(
    formula="""
        crime_100m ~ treatment + post + treatment_x_post
        + baseline_crime_intensity
    """,
    data=panel
).fit(
    cov_type="cluster",
    cov_kwds={"groups": panel["pair_id"]}
)

print(did_100m.summary())

                            OLS Regression Results                            
Dep. Variable:             crime_100m   R-squared:                       0.169
Model:                            OLS   Adj. R-squared:                  0.168
Method:                 Least Squares   F-statistic:                     20.54
Date:                Wed, 23 Sep 2026   Prob (F-statistic):           6.32e-17
Time:                        10:36:50   Log-Likelihood:             1.7227e+05
No. Observations:              276744   AIC:                        -3.445e+05
Df Residuals:                  276739   BIC:                        -3.445e+05
Df Model:                           4                                         
Covariance Type:              cluster                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [10]:
# Difference-in-Differences model for 100–250m displacement crime

did_250m = smf.ols(
    formula="""
        crime_250m ~ treatment + post + treatment_x_post
        + baseline_crime_intensity
    """,
    data=panel
).fit(
    cov_type="cluster",
    cov_kwds={"groups": panel["pair_id"]}
)

print(did_250m.summary())

                            OLS Regression Results                            
Dep. Variable:             crime_250m   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     7.659
Date:                Wed, 23 Sep 2026   Prob (F-statistic):           3.65e-06
Time:                        10:37:56   Log-Likelihood:            -1.6019e+05
No. Observations:              276744   AIC:                         3.204e+05
Df Residuals:                  276739   BIC:                         3.205e+05
Df Model:                           4                                         
Covariance Type:              cluster                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [11]:
# Extract the DiD interaction results from both models

did_results = pd.DataFrame({
    "Outcome": [
        "Crime 0–100m",
        "Crime 100–250m"
    ],
    "DiD_Coefficient": [
        did_100m.params["treatment_x_post"],
        did_250m.params["treatment_x_post"]
    ],
    "Std_Error": [
        did_100m.bse["treatment_x_post"],
        did_250m.bse["treatment_x_post"]
    ],
    "P_Value": [
        did_100m.pvalues["treatment_x_post"],
        did_250m.pvalues["treatment_x_post"]
    ],
    "CI_Lower": [
        did_100m.conf_int().loc["treatment_x_post", 0],
        did_250m.conf_int().loc["treatment_x_post", 0]
    ],
    "CI_Upper": [
        did_100m.conf_int().loc["treatment_x_post", 1],
        did_250m.conf_int().loc["treatment_x_post", 1]
    ]
})

print(did_results.to_string(index=False))

       Outcome  DiD_Coefficient  Std_Error  P_Value  CI_Lower  CI_Upper
  Crime 0–100m         0.000141   0.000796 0.859390 -0.001418  0.001700
Crime 100–250m        -0.001333   0.002266 0.556336 -0.005776  0.003109


In [12]:
# Add conventional significance indicators for reporting

def significance_stars(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    else:
        return ""

did_results["Significance"] = did_results["P_Value"].apply(significance_stars)

print(did_results.to_string(index=False))

       Outcome  DiD_Coefficient  Std_Error  P_Value  CI_Lower  CI_Upper Significance
  Crime 0–100m         0.000141   0.000796 0.859390 -0.001418  0.001700             
Crime 100–250m        -0.001333   0.002266 0.556336 -0.005776  0.003109             


In [13]:
# Compare treatment and control crime levels during the PRE period

pre_panel = panel[panel["period"] == "pre"].copy()

pre_balance = (
    pre_panel
    .groupby("treatment")[["crime_100m", "crime_250m", "baseline_crime_intensity"]]
    .agg(["mean", "std", "median", "count"])
)

print("PRE-period treatment/control balance:")
print(pre_balance.to_string())

PRE-period treatment/control balance:
          crime_100m                         crime_250m                         baseline_crime_intensity                        
                mean       std median  count       mean       std median  count                     mean       std median  count
treatment                                                                                                                       
0           0.006071  0.112554    0.0  46124   0.023372  0.222277    0.0  46124                 0.006071  0.112554    0.0  46124
1           0.004510  0.087739    0.0  46124   0.024001  0.231153    0.0  46124                 0.004510  0.087739    0.0  46124


In [14]:
# Calculate treatment minus control differences in PRE-period means

pre_means = (
    pre_panel
    .groupby("treatment")[["crime_100m", "crime_250m"]]
    .mean()
)

pre_difference = pre_means.loc[1] - pre_means.loc[0]

print("PRE-period mean differences (Treatment - Control):")
print(pre_difference)

PRE-period mean differences (Treatment - Control):
crime_100m   -0.001561
crime_250m    0.000629
dtype: float64


In [15]:
# Descriptive crime means across all three periods

period_summary = (
    panel
    .groupby(["treatment", "period"])[
        ["crime_100m", "crime_250m"]
    ]
    .agg(["mean", "std", "count"])
    .reset_index()
)

print("Crime outcomes by treatment group and period:")
print(period_summary.to_string(index=False))

Crime outcomes by treatment group and period:
treatment period crime_100m                crime_250m               
                       mean      std count       mean      std count
        0 during   0.008759 0.266122 46124   0.036510 0.734110 46124
        0   post   0.005117 0.093682 46124   0.022440 0.230465 46124
        0    pre   0.006071 0.112554 46124   0.023372 0.222277 46124
        1 during   0.006157 0.121622 46124   0.035621 0.609789 46124
        1   post   0.004878 0.083021 46124   0.021919 0.230232 46124
        1    pre   0.004510 0.087739 46124   0.024001 0.231153 46124


In [16]:
# Clean mean table for reporting

period_means = (
    panel
    .groupby(["treatment", "period"])[
        ["crime_100m", "crime_250m"]
    ]
    .mean()
    .reset_index()
)

period_means["Group"] = period_means["treatment"].map({
    0: "Control",
    1: "Treatment"
})

period_means = period_means[
    ["Group", "period", "crime_100m", "crime_250m"]
]

print(period_means.to_string(index=False))

    Group period  crime_100m  crime_250m
  Control during    0.008759    0.036510
  Control   post    0.005117    0.022440
  Control    pre    0.006071    0.023372
Treatment during    0.006157    0.035621
Treatment   post    0.004878    0.021919
Treatment    pre    0.004510    0.024001


In [17]:
# Calculate raw Difference-in-Differences from group means

mean_lookup = (
    panel
    .groupby(["treatment", "period"])[["crime_100m", "crime_250m"]]
    .mean()
)

# PRE -> DURING
did_during_100m = (
    (mean_lookup.loc[(1, "during"), "crime_100m"]
     - mean_lookup.loc[(1, "pre"), "crime_100m"])
    -
    (mean_lookup.loc[(0, "during"), "crime_100m"]
     - mean_lookup.loc[(0, "pre"), "crime_100m"])
)

did_during_250m = (
    (mean_lookup.loc[(1, "during"), "crime_250m"]
     - mean_lookup.loc[(1, "pre"), "crime_250m"])
    -
    (mean_lookup.loc[(0, "during"), "crime_250m"]
     - mean_lookup.loc[(0, "pre"), "crime_250m"])
)

# PRE -> POST
did_post_100m = (
    (mean_lookup.loc[(1, "post"), "crime_100m"]
     - mean_lookup.loc[(1, "pre"), "crime_100m"])
    -
    (mean_lookup.loc[(0, "post"), "crime_100m"]
     - mean_lookup.loc[(0, "pre"), "crime_100m"])
)

did_post_250m = (
    (mean_lookup.loc[(1, "post"), "crime_250m"]
     - mean_lookup.loc[(1, "pre"), "crime_250m"])
    -
    (mean_lookup.loc[(0, "post"), "crime_250m"]
     - mean_lookup.loc[(0, "pre"), "crime_250m"])
)

raw_did = pd.DataFrame({
    "Outcome": ["Crime 0–100m", "Crime 100–250m"],
    "PRE_to_DURING_DiD": [
        did_during_100m,
        did_during_250m
    ],
    "PRE_to_POST_DiD": [
        did_post_100m,
        did_post_250m
    ]
})

print(raw_did.to_string(index=False))


       Outcome  PRE_to_DURING_DiD  PRE_to_POST_DiD
  Crime 0–100m          -0.001041         0.001323
Crime 100–250m          -0.001518        -0.001149


In [18]:
# Create separate indicators for DURING and POST periods

panel["during"] = (panel["period"] == "during").astype(int)
panel["post_period"] = (panel["period"] == "post").astype(int)

# Treatment interactions
panel["treatment_x_during"] = (
    panel["treatment"] * panel["during"]
)

panel["treatment_x_post_period"] = (
    panel["treatment"] * panel["post_period"]
)

print(
    panel[
        [
            "period",
            "treatment",
            "during",
            "post_period",
            "treatment_x_during",
            "treatment_x_post_period"
        ]
    ].drop_duplicates()
    .sort_values(["treatment", "period"])
    .to_string(index=False)
)

period  treatment  during  post_period  treatment_x_during  treatment_x_post_period
during          0       1            0                   0                        0
  post          0       0            1                   0                        0
   pre          0       0            0                   0                        0
during          1       1            0                   1                        0
  post          1       0            1                   0                        1
   pre          1       0            0                   0                        0


In [19]:
# Separate-period DiD model for 0–100m crime

did_100m_separate = smf.ols(
    formula="""
        crime_100m ~ treatment
        + during
        + post_period
        + treatment_x_during
        + treatment_x_post_period
        + baseline_crime_intensity
    """,
    data=panel
).fit(
    cov_type="cluster",
    cov_kwds={"groups": panel["pair_id"]}
)

print(did_100m_separate.summary())

                            OLS Regression Results                            
Dep. Variable:             crime_100m   R-squared:                       0.169
Model:                            OLS   Adj. R-squared:                  0.169
Method:                 Least Squares   F-statistic:                     14.98
Date:                Wed, 23 Sep 2026   Prob (F-statistic):           3.38e-17
Time:                        10:45:09   Log-Likelihood:             1.7228e+05
No. Observations:              276744   AIC:                        -3.445e+05
Df Residuals:                  276737   BIC:                        -3.445e+05
Df Model:                           6                                         
Covariance Type:              cluster                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [20]:
# Separate-period DiD model for 100–250m displacement crime

did_250m_separate = smf.ols(
    formula="""
        crime_250m ~ treatment
        + during
        + post_period
        + treatment_x_during
        + treatment_x_post_period
        + baseline_crime_intensity
    """,
    data=panel
).fit(
    cov_type="cluster",
    cov_kwds={"groups": panel["pair_id"]}
)

print(did_250m_separate.summary())

                            OLS Regression Results                            
Dep. Variable:             crime_250m   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     8.667
Date:                Wed, 23 Sep 2026   Prob (F-statistic):           1.88e-09
Time:                        10:46:47   Log-Likelihood:            -1.6017e+05
No. Observations:              276744   AIC:                         3.204e+05
Df Residuals:                  276737   BIC:                         3.204e+05
Df Model:                           6                                         
Covariance Type:              cluster                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [21]:
# Final Stage 9 DiD results table
# Separate DURING and POST estimates

final_did_results = pd.DataFrame({
    "Outcome": [
        "Crime 0–100m",
        "Crime 0–100m",
        "Crime 100–250m",
        "Crime 100–250m"
    ],
    "Period": [
        "DURING",
        "POST",
        "DURING",
        "POST"
    ],
    "Coefficient": [
        did_100m_separate.params["treatment_x_during"],
        did_100m_separate.params["treatment_x_post_period"],
        did_250m_separate.params["treatment_x_during"],
        did_250m_separate.params["treatment_x_post_period"]
    ],
    "Std_Error": [
        did_100m_separate.bse["treatment_x_during"],
        did_100m_separate.bse["treatment_x_post_period"],
        did_250m_separate.bse["treatment_x_during"],
        did_250m_separate.bse["treatment_x_post_period"]
    ],
    "P_Value": [
        did_100m_separate.pvalues["treatment_x_during"],
        did_100m_separate.pvalues["treatment_x_post_period"],
        did_250m_separate.pvalues["treatment_x_during"],
        did_250m_separate.pvalues["treatment_x_post_period"]
    ],
    "CI_Lower": [
        did_100m_separate.conf_int().loc["treatment_x_during", 0],
        did_100m_separate.conf_int().loc["treatment_x_post_period", 0],
        did_250m_separate.conf_int().loc["treatment_x_during", 0],
        did_250m_separate.conf_int().loc["treatment_x_post_period", 0]
    ],
    "CI_Upper": [
        did_100m_separate.conf_int().loc["treatment_x_during", 1],
        did_100m_separate.conf_int().loc["treatment_x_post_period", 1],
        did_250m_separate.conf_int().loc["treatment_x_during", 1],
        did_250m_separate.conf_int().loc["treatment_x_post_period", 1]
    ]
})

print(final_did_results.to_string(index=False))

       Outcome Period  Coefficient  Std_Error  P_Value  CI_Lower  CI_Upper
  Crime 0–100m DURING    -0.001041   0.001261 0.409162 -0.003512  0.001431
  Crime 0–100m   POST     0.001323   0.000646 0.040538  0.000057  0.002588
Crime 100–250m DURING    -0.001518   0.003799 0.689532 -0.008963  0.005928
Crime 100–250m   POST    -0.001149   0.001259 0.361473 -0.003617  0.001319


In [22]:
# Save Stage 9 DiD results

RESULTS_FILE = PROCESSED_DIR / "did_results.csv"

final_did_results.to_csv(
    RESULTS_FILE,
    index=False
)

print(f"DiD results saved to: {RESULTS_FILE}")

DiD results saved to: ..\data\processed\did_results.csv


In [23]:
# Final Stage 9 model specification

print("""
FINAL DIFFERENCE-IN-DIFFERENCES SPECIFICATION
----------------------------------------------

Reference period:
    PRE

Period indicators:
    during
    post_period

Treatment indicator:
    treatment

Interaction terms:
    treatment_x_during
    treatment_x_post_period

Covariate:
    baseline_crime_intensity

Outcomes:
    crime_100m
    crime_250m

Standard errors:
    Clustered by pair_id

Interpretation:
    treatment_x_during compares the treatment-control change
    from PRE to DURING.

    treatment_x_post_period compares the treatment-control change
    from PRE to POST.
""")


FINAL DIFFERENCE-IN-DIFFERENCES SPECIFICATION
----------------------------------------------

Reference period:
    PRE

Period indicators:
    during
    post_period

Treatment indicator:
    treatment

Interaction terms:
    treatment_x_during
    treatment_x_post_period

Covariate:
    baseline_crime_intensity

Outcomes:
    crime_100m
    crime_250m

Standard errors:
    Clustered by pair_id

Interpretation:
    treatment_x_during compares the treatment-control change
    from PRE to DURING.

    treatment_x_post_period compares the treatment-control change
    from PRE to POST.



In [24]:
# Verify saved Stage 9 results

print("Results file:", RESULTS_FILE)
print("File exists:", RESULTS_FILE.exists())

saved_results = pd.read_csv(RESULTS_FILE)

print("\nSaved results:")
print(saved_results.to_string(index=False))

Results file: ..\data\processed\did_results.csv
File exists: True

Saved results:
       Outcome Period  Coefficient  Std_Error  P_Value  CI_Lower  CI_Upper
  Crime 0–100m DURING    -0.001041   0.001261 0.409162 -0.003512  0.001431
  Crime 0–100m   POST     0.001323   0.000646 0.040538  0.000057  0.002588
Crime 100–250m DURING    -0.001518   0.003799 0.689532 -0.008963  0.005928
Crime 100–250m   POST    -0.001149   0.001259 0.361473 -0.003617  0.001319


## Stage 9 Summary

The Difference-in-Differences analysis was performed using the balanced
causal panel created in Stage 8.

### Model specification

- Reference period: PRE
- Treatment group: outage locations
- Control group: matched control locations
- Period indicators: DURING and POST
- Treatment interactions:
  - treatment × DURING
  - treatment × POST
- Covariate: baseline crime intensity
- Standard errors: clustered by treatment-control pair

### Outcomes

1. Crime within 0–100m
2. Crime within the 100–250m displacement ring

The final period-specific DiD estimates were saved to:

`data/processed/did_results.csv`

A conventional parallel-trends test could not be performed because the
causal panel contains only one PRE window.